## Khám phá Dữ liệu (EDA - Exploratory Data Analysis)

In [ ]:
# Phân bố biến mục tiêu
plt.figure(figsize=(10, 6))
sns.histplot(full_data['avg_rating'], bins=20, kde=True)
plt.title('Phân bố điểm rating của người dùng MovieLens')
plt.xlabel('Rating')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()


In [ ]:
# 2. Thống kê mô tả
print(full_data['Rating'].describe())
print(f"Skewness: {full_data['Rating'].skew():.3f}")
print(f"Kurtosis: {full_data['Rating'].kurtosis():.3f}")


In [ ]:
# 3. Phân bố các biến numerical khác
numerical_cols = [
    'budget', 
    'revenue', 
    'runtime', 
    'vote_average', 
    'vote_count',
    'popularity',
    'director_power',
    'actor_power'
]

full_data[numerical_cols].hist(figsize=(15, 10), bins=30)
plt.suptitle('Phân bố các biến định lượng', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 4. Tần suất của các biến phân loại
categorical_cols = ['Genres']

for col in categorical_cols:
    top_values = full_data[col].value_counts().head(10).reset_index()
    top_values.columns = [col, 'count']

    plt.figure(figsize=(8, 5))
    sns.barplot(data=top_values, x=col, y='count')
    plt.title(f'Top 10 {col} phổ biến nhất')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

### Phân tích hai biến

Insights cần tìm:
+ Biến nào có correlation mạnh với rating?
+ Có non-linear relationships không?

In [ ]:
plt.figure(figsize=(12, 10))

corr_cols = numerical_cols + ['avg_rating']
df_movie_scope = full_data.drop_duplicates(subset=['MovieID']).copy()
correlation_matrix = df_movie_scope[corr_cols].corr()

sns.heatmap(
    correlation_matrix,
    annot=True,
    fmt=".2f",
    cmap='coolwarm',
    center=0
)

plt.title('Ma trận tương quan giữa các biến số và Rating')
plt.tight_layout()
plt.show()


In [ ]:
# 1. Phân loại Đạo diễn thành 4 đẳng cấp (Quartiles)
# Low: Vô danh, Medium: Triển vọng, High: Nổi tiếng, Legendary: Huyền thoại
full_data['director_tier'] = pd.qcut(full_data['director_power'], q=4, labels=['Low', 'Medium', 'High', 'Legendary'])

# 2. Tính toán "Tỷ lệ thảm họa" (Flop Rate) cho từng nhóm
# Giả sử phim "thảm họa" là phim có rating < 2.5
flop_threshold = 2.5
flop_analysis = full_data.groupby('director_tier')['avg_rating'].apply(lambda x: (x < flop_threshold).mean() * 100).reset_index()
flop_analysis.columns = ['Director Tier', 'Flop Rate (%)']

print("Tỷ lệ bị chấm điểm thấp (Rating < 2.5) theo đẳng cấp đạo diễn:")
print(flop_analysis)

# 3. Trực quan hóa "Lưới an toàn" (Safety Net) bằng Boxplot
plt.figure(figsize=(10, 6))
sns.boxplot(x='director_tier', y='avg_rating', data=full_data, palette="viridis")

plt.title('Phân phối điểm số theo đẳng cấp Đạo diễn: Có tồn tại "Lưới an toàn"?')
plt.ylabel('Điểm đánh giá (Rating)')
plt.xlabel('Đẳng cấp Đạo diễn')
plt.axhline(y=2.5, color='r', linestyle='--', label='Ngưỡng thảm họa (2.5)')
plt.legend()
plt.show()



In [ ]:
# 4. Phân tích độ lệch chuẩn (Consistency)
# Xem ai ổn định hơn: Người xem có xu hướng chấm điểm "đồng lòng" cho Legendary không?
std_analysis = full_data.groupby('director_tier')['avg_rating'].std().reset_index()
std_analysis.columns = ['Director Tier', 'Rating Std Dev']
print("\nĐộ lệch chuẩn điểm số (Sự nhất quán trong đánh giá):")
print(std_analysis)

In [ ]:
full_data['runtime_group'] = pd.cut(
    full_data['runtime'],
    bins=[0, 90, 120, 150, 300],
    labels=['Ngắn', 'Vừa', 'Dài', 'Rất dài']
)

plt.figure(figsize=(8, 6))
sns.boxplot(
    data=full_data,
    x='runtime_group',
    y='Rating'
)

plt.title('Rating theo nhóm thời lượng phim')
plt.xlabel('Nhóm thời lượng')
plt.ylabel('Rating')
plt.tight_layout()
plt.show()


In [ ]:
bins = [0, 18, 25, 35, 50, 100]
labels = ['<18', '18–25', '26–35', '36–50', '50+']

users['age_group'] = pd.cut(
    users['Age'],
    bins=bins,
    labels=labels,
    right=False
)

full_data = full_data.merge(
    users[['UserID', 'Gender', 'age_group']],
    on='UserID',
    how='left'
)

plt.figure(figsize=(8, 6))
sns.violinplot(
    data=full_data,
    x='age_group',
    y='Rating'
)

plt.title('Phân bố rating theo nhóm tuổi người dùng')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(6, 5))
sns.boxplot(
    data=full_data,
    x='Gender',
    y='Rating'
)

plt.title('Rating theo giới tính')
plt.tight_layout()
plt.show()



In [ ]:
genre_df = full_data[['Genres', 'Rating']].copy()
genre_df['Genres'] = genre_df['Genres'].str.split('|')
genre_df = genre_df.explode('Genres')

genre_rating = genre_df.groupby('Genres')['Rating'].agg(
    mean='mean',
    count='count',
    std='std'
)

genre_rating = genre_rating[genre_rating['count'] > 1000] \
    .sort_values('mean', ascending=False) \
    .head(15)

plt.figure(figsize=(12, 6))
sns.barplot(
    x=genre_rating.index,
    y=genre_rating['mean']
)

plt.xticks(rotation=45, ha='right')
plt.ylabel('Average Rating')
plt.title('Rating trung bình theo thể loại (Top 15)')
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1,2, figsize=(16,6))

sns.regplot(x='budget', y='avg_rating', data=df_movie_scope, ax=axes[0], scatter_kws={'alpha':0.3}, line_kws={'color':'red'})
axes[0].set_xscale('log')
axes[0].set_title('Ngân sách và Đánh giá')

sns.regplot(x='revenue', y='avg_rating', data=df_movie_scope, ax=axes[1], scatter_kws={'alpha':0.3}, line_kws={'color':'green'})
axes[1].set_xscale('log')
axes[1].set_title('Doanh thu và Đánh giá')

plt.tight_layout()
plt.show()

### Phân tích đa biến

Insights cần tìm:
+ Có interaction effects không? (VD: budget cao + action genre = rating cao)

In [ ]:
# 1. Pairplot cho top features
top_features = ['budget', 'runtime', 'popularity', 'avg_rating', 'director_power', 'actor_power']
sns.pairplot(full_data[top_features], diag_kind='kde', plot_kws={'alpha': 0.5})
plt.show()

In [ ]:
from mpl_toolkits.mplot3d import Axes3D
import numpy as np

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

ax.scatter(
    np.log10(full_data['popularity'] + 1),
    full_data['runtime'],
    full_data['avg_rating'],
    alpha=0.4
)

ax.set_xlabel('log(Popularity)')
ax.set_ylabel('Runtime')
ax.set_zlabel('Rating')

plt.title('Tương tác giữa Popularity, Runtime và Rating')
plt.tight_layout()
plt.show()


In [ ]:
# 2. Rating theo genre và budget_category
# Tạo budget_category
full_data['budget_category'] = pd.qcut(
    full_data['budget'],
    q=4,
    labels=['Thấp', 'Trung bình', 'Cao', 'Rất cao']
)

# Xử lý genres
genre_budget_df = full_data[['Genres', 'budget_category', 'avg_rating']].copy()
genre_budget_df['Genres'] = genre_budget_df['Genres'].str.split('|')
genre_budget_df = genre_budget_df.explode('Genres')

pivot_table = genre_budget_df.pivot_table(
    values='avg_rating',
    index='Genres',
    columns='budget_category',
    aggfunc='mean'
)

plt.figure(figsize=(12, 8))
sns.heatmap(
    pivot_table,
    cmap='YlOrRd',
    annot=False
)

plt.title('Rating trung bình theo Thể loại và Nhóm ngân sách')
plt.ylabel('Thể loại')
plt.xlabel('Nhóm ngân sách')
plt.tight_layout()
plt.show()


In [ ]:
from scipy import stats

# Lọc dữ liệu hợp lệ
gender_df = full_data[['Gender', 'avg_rating']].dropna()

male_ratings = gender_df[gender_df['Gender'] == 'M']['avg_rating']
female_ratings = gender_df[gender_df['Gender'] == 'F']['avg_rating']

# Welch's t-test (equal_var=False)
t_stat, p_value = stats.ttest_ind(
    male_ratings,
    female_ratings,
    equal_var=False
)

print("T-TEST: Rating giữa Nam và Nữ")
print(f"T-statistic = {t_stat:.4f}")
print(f"P-value     = {p_value:.4f}")

if p_value < 0.05:
    print("✓ Có sự khác biệt có ý nghĩa thống kê giữa rating của nam và nữ")
else:
    print("✗ Không có sự khác biệt có ý nghĩa thống kê")


In [ ]:
from scipy import stats

# Tạo biến high_rating
full_data['high_rating'] = full_data['avg_rating'] >= 4.0

# Tách genres thành nhiều dòng
genre_df = full_data[['Genres', 'high_rating']].dropna()
genre_df = genre_df.assign(genres=genre_df['Genres'].str.split('|')).explode('Genres')

# Chỉ giữ genre có đủ mẫu
genre_counts = genre_df['Genres'].value_counts()
valid_genres = genre_counts[genre_counts > 1000].index

genre_df = genre_df[genre_df['Genres'].isin(valid_genres)]

contingency_table = pd.crosstab(
    genre_df['Genres'],
    genre_df['high_rating']
)

chi2, p_value, dof, expected = stats.chi2_contingency(contingency_table)

print("CHI-SQUARE TEST: Genre vs High Rating")
print(f"Chi-square statistic = {chi2:.4f}")
print(f"P-value             = {p_value:.4e}")
print(f"Degrees of freedom  = {dof}")

if p_value < 0.05:
    print("✓ Genre và high rating KHÔNG độc lập (có mối liên hệ)")
else:
    print("✗ Không đủ bằng chứng cho thấy genre ảnh hưởng đến high rating")


In [ ]:
genre_success_rate = (
    genre_df
    .groupby('Genres')['high_rating']
    .mean()
    .sort_values(ascending=False)
)

genre_success_rate.head(10)


In [ ]:
from scipy.stats import pointbiserialr

print("Point-biserial correlation (Genre vs Rating):")

genres_to_test = ['Action', 'Drama', 'Comedy']

for genre in genres_to_test:
    full_data[f'is_{genre}'] = full_data['Genres'].str.contains(genre, na=False).astype(int)

    corr, p_value = pointbiserialr(
        full_data[f'is_{genre}'],
        full_data['avg_rating']
    )

    print(f"{genre:10s} | r = {corr:.3f} | p-value = {p_value:.4e}")


In [ ]:
# Chi-square test: Genre và high rating (>4.0) có độc lập không?
df = full_data.copy()
df['high_rating'] = df['avg_rating'] >= 4.0
contingency_table = pd.crosstab(df['Genres'], df['high_rating'])

chi2, p_value, dof, expected = stats.chi2_contingency(contingency_table)
print(f"Chi-square: {chi2:.4f}")
print(f"P-value: {p_value:.4f}")

In [ ]:
import numpy as np

n = contingency_table.values.sum()
cramers_v = np.sqrt(
    chi2 / (n * (min(contingency_table.shape) - 1))
)

print(f"Cramér's V = {cramers_v:.4f}")


In [ ]:
import pandas as pd
from scipy.stats import pointbiserialr

# ---- PHÒNG THỦ TRÙNG CỘT ----
num_cols_corr = list(dict.fromkeys(numerical_cols))
if 'vote_average' in num_cols_corr:
    num_cols_corr.remove('vote_average')

# ---- 1. Pearson correlation ----
corr_p = df[num_cols_corr + ['vote_average']].corr(method='pearson')

vote_corr_p = corr_p.loc[:, 'vote_average']
if isinstance(vote_corr_p, pd.DataFrame):
    vote_corr_p = vote_corr_p.iloc[:, 0]

pearson_corr = vote_corr_p.sort_values(ascending=False)

print("Pearson Correlation với Vote Average:")
print(pearson_corr)


# ---- 2. Spearman correlation ----
corr_s = df[num_cols_corr + ['vote_average']].corr(method='spearman')

vote_corr_s = corr_s.loc[:, 'vote_average']
if isinstance(vote_corr_s, pd.DataFrame):
    vote_corr_s = vote_corr_s.iloc[:, 0]

spearman_corr = vote_corr_s.sort_values(ascending=False)

print("\nSpearman Correlation với Vote Average:")
print(spearman_corr)


# ---- 3. Point-biserial correlation (Genre vs Vote Average) ----
print("\nPoint-biserial correlation (Genre vs Vote Average):")

for genre in ['Action', 'Drama', 'Comedy']:
    df[f'is_{genre}'] = df['Genres'].str.contains(genre, na=False).astype(int)
    corr, p_value = pointbiserialr(df[f'is_{genre}'], df['vote_average'])
    print(f"{genre:10s} | r = {corr:.3f} | p-value = {p_value:.4e}")


In [ ]:
import numpy as np

# Cohen's d (Welch version – không giả định phương sai bằng nhau)
def cohens_d_welch(group1, group2):
    mean1, mean2 = group1.mean(), group2.mean()
    var1, var2 = group1.var(ddof=1), group2.var(ddof=1)
    n1, n2 = len(group1), len(group2)

    pooled_std = np.sqrt((var1 + var2) / 2)
    return (mean1 - mean2) / pooled_std


# Lọc dữ liệu hợp lệ
gender_df = full_data[['Gender', 'vote_average']].dropna()

male_ratings = gender_df[gender_df['Gender'] == 'M']['vote_average']
female_ratings = gender_df[gender_df['Gender'] == 'F']['vote_average']

d = cohens_d_welch(male_ratings, female_ratings)

print(f"Cohen's d: {d:.3f}")

# Diễn giải magnitude
if abs(d) < 0.2:
    interpretation = "negligible"
elif abs(d) < 0.5:
    interpretation = "small"
elif abs(d) < 0.8:
    interpretation = "medium"
else:
    interpretation = "large"

print(f"Interpretation: {interpretation}")
